In [ ]:
import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors
from typing import List, Dict, Tuple, Optional, Set
from dataclasses import dataclass
import pickle
import logging
import os
import time

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class SemanticPath:
    start_word: str
    end_word: str
    path: List[str]
    total_distance: float
    path_strength: float
    hops: int

@dataclass
class ClueCandidate:
    clue_word: str
    target_words: List[str]
    paths: Dict[str, SemanticPath]
    avg_path_strength: float
    coverage: int
    safety_score: float
    creativity_score: float
    final_score: float

class Vocabulary:

    OFFICIAL_WORDS = [
        'africa', 'agent', 'air', 'alien', 'alps', 'amazon', 'ambulance', 'america',
        'angel', 'antarctica', 'apple', 'arm', 'atlantis', 'australia', 'back', 'ball',
        'band', 'bank', 'bar', 'bark', 'bat', 'battery', 'beach', 'bear', 'beat', 'bed',
        'beijing', 'bell', 'belt', 'berlin', 'bermuda', 'berry', 'bill', 'block', 'board',
        'bolt', 'bomb', 'bond', 'boom', 'boot', 'bottle', 'bow', 'box', 'bridge', 'brush',
        'buck', 'buffalo', 'bug', 'bugle', 'button', 'calf', 'canada', 'cap', 'capital',
        'car', 'card', 'carrot', 'casino', 'cast', 'cat', 'cell', 'centaur', 'center',
        'chair', 'change', 'charge', 'check', 'chest', 'chick', 'china', 'chocolate',
        'church', 'circle', 'cliff', 'cloak', 'club', 'code', 'cold', 'comic', 'compound',
        'concert', 'conductor', 'contract', 'cook', 'copper', 'cotton', 'court', 'cover',
        'crane', 'crash', 'cricket', 'cross', 'crown', 'cycle', 'czech', 'dance', 'date',
        'day', 'death', 'deck', 'degree', 'diamond', 'dice', 'dinosaur', 'disease',
        'doctor', 'dog', 'draft', 'dragon', 'dress', 'drill', 'drop', 'duck', 'dwarf',
        'eagle', 'egypt', 'embassy', 'engine', 'england', 'europe', 'eye', 'face', 'fair',
        'fall', 'fan', 'fence', 'field', 'fighter', 'figure', 'file', 'film', 'fire',
        'fish', 'flute', 'fly', 'foot', 'force', 'forest', 'fork', 'france', 'game',
        'gas', 'genius', 'germany', 'ghost', 'giant', 'glass', 'glove', 'gold', 'grace',
        'grass', 'greece', 'green', 'ground', 'ham', 'hand', 'hawk', 'head', 'heart',
        'helicopter', 'hole', 'hollywood', 'honey', 'hood', 'hook', 'horn', 'horse',
        'horseshoe', 'hospital', 'hotel', 'ice', 'india', 'iron', 'ivory', 'jack', 'jam',
        'jet', 'jupiter', 'kangaroo', 'ketchup', 'key', 'kid', 'king', 'kiwi', 'knife',
        'knight', 'lab', 'lap', 'laser', 'lawyer', 'lead', 'lemon', 'leprechaun', 'life',
        'light', 'limousine', 'line', 'link', 'lion', 'litter', 'lock', 'log', 'london',
        'luck', 'mail', 'mammoth', 'maple', 'marble', 'march', 'mass', 'match', 'mercury',
        'mexico', 'microscope', 'millionaire', 'mine', 'mint', 'missile', 'model', 'mole',
        'moon', 'moscow', 'mount', 'mouse', 'mouth', 'mug', 'nail', 'needle', 'net',
        'night', 'ninja', 'note', 'novel', 'nurse', 'nut', 'octopus', 'oil', 'olive',
        'opera', 'orange', 'organ', 'palm', 'pan', 'pants', 'paper', 'parachute', 'park',
        'part', 'pass', 'paste', 'penguin', 'phoenix', 'piano', 'pie', 'pilot', 'pin',
        'pipe', 'pirate', 'pistol', 'pit', 'pitch', 'plane', 'plastic', 'plate',
        'platypus', 'play', 'plot', 'point', 'poison', 'pole', 'police', 'pool', 'port',
        'post', 'pound', 'press', 'princess', 'pumpkin', 'pupil', 'pyramid', 'queen',
        'rabbit', 'racket', 'ray', 'revolution', 'ring', 'robin', 'robot', 'rock', 'rome',
        'root', 'rose', 'roulette', 'round', 'row', 'ruler', 'satellite', 'saturn',
        'scale', 'school', 'scientist', 'scorpion', 'screen', 'scuba', 'diver', 'seal',
        'server', 'shadow', 'shakespeare', 'shark', 'ship', 'shoe', 'shop', 'shot',
        'sink', 'skyscraper', 'slip', 'slug', 'smuggler', 'snow', 'snowman', 'sock',
        'soldier', 'soul', 'sound', 'space', 'spell', 'spider', 'spike', 'spine', 'spot',
        'spring', 'spy', 'square', 'stadium', 'staff', 'star', 'state', 'stick', 'stock',
        'straw', 'stream', 'strike', 'string', 'sub', 'suit', 'superhero', 'swing',
        'switch', 'table', 'tablet', 'tag', 'tail', 'tap', 'teacher', 'telescope',
        'temple', 'theater', 'thief', 'thumb', 'tick', 'tie', 'time', 'tokyo', 'tooth',
        'torch', 'tower', 'track', 'train', 'triangle', 'trip', 'trunk', 'tube', 'turkey',
        'undertaker', 'unicorn', 'vacuum', 'van', 'vet', 'wake', 'wall', 'war', 'washer',
        'washington', 'watch', 'water', 'wave', 'web', 'well', 'whale', 'whip', 'wind',
        'witch', 'worm', 'yard'
    ]

    def __init__(self, wordlist_file: str = None):
        self.official_words = self._load_official_words(wordlist_file)
        self.expanded_words = set()

    def _load_official_words(self, wordlist_file: str = None) -> Set[str]:
        if wordlist_file and os.path.exists(wordlist_file):
            logger.info(f"Loading official words from {wordlist_file}")
            words = set()
            with open(wordlist_file, 'r') as f:
                for line in f:
                    word = line.strip().lower()
                    if word and len(word) >= 2:
                        words.add(word)
            logger.info(f"Loaded {len(words)} official words")
            return words
        else:
            return set(self.OFFICIAL_WORDS)

    def get_board_words(self, n: int = 25) -> List[str]:
        import random
        word_list = list(self.official_words)
        if len(word_list) < n:
            raise ValueError(f"Need at least {n} words, only have {len(word_list)}")
        return random.sample(word_list, n)

    def expand_vocabulary(self, embeddings, target_size: int = 7000) -> Set[str]:
        logger.info(f"Expanding vocabulary from {len(self.official_words)} to ~{target_size} words")
        expanded = set(self.official_words)
        for word in self.official_words:
            if word in embeddings:
                similar = self._find_similar_words(word, embeddings, n=10)
                for similar_word, similarity in similar:
                    if similarity > 0.3 and self._is_valid_word(similar_word):
                        expanded.add(similar_word)
                        if len(expanded) >= target_size:
                            break
            if len(expanded) >= target_size:
                break
        logger.info(f"Expanded vocabulary to {len(expanded)} words")
        self.expanded_words = expanded
        return expanded

    def _find_similar_words(self, word, embeddings, n=10):
        if word not in embeddings:
            return []
        target_vec = embeddings[word]
        similarities = []
        vocab_sample = list(embeddings.keys())[:15000]
        for vocab_word in vocab_sample:
            if vocab_word != word:
                vec = embeddings[vocab_word]
                dot_product = np.dot(target_vec, vec)
                norm1 = np.linalg.norm(target_vec)
                norm2 = np.linalg.norm(vec)
                if norm1 > 0 and norm2 > 0:
                    similarity = dot_product / (norm1 * norm2)
                    similarities.append((vocab_word, similarity))
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:n]

    def _is_valid_word(self, word):
        if len(word) < 3 or len(word) > 12:
            return False
        if not word.isalpha():
            return False
        if word in {'the', 'and', 'or', 'but', 'a', 'an', 'is', 'are', 'was', 'were'}:
            return False
        return True


class Embeddings:
    def __init__(self):
        self.embeddings = {}
        self.vocab = set()
        self.vector_dim = None

    def load_glove(self, filepath, max_vocab=20000):
        if not os.path.exists(filepath):
            logger.error(f"GloVe file not found: {filepath}")
            return False
        logger.info(f"Loading GloVe embeddings from {filepath}")
        start_time = time.time()
        loaded = 0
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f):
                    if line_num % 50000 == 0 and line_num > 0:
                        elapsed = time.time() - start_time
                        logger.info(f"Processed {line_num:,} lines, loaded {loaded:,} words ({elapsed:.1f}s)")
                    parts = line.strip().split()
                    if len(parts) < 50:
                        continue
                    word = parts[0].lower()
                    try:
                        vector = np.array(parts[1:], dtype=np.float32)
                        if self.vector_dim is None:
                            self.vector_dim = len(vector)
                        if len(vector) == self.vector_dim:
                            self.embeddings[word] = vector
                            self.vocab.add(word)
                            loaded += 1
                            if loaded >= max_vocab:
                                break
                    except (ValueError, TypeError):
                        continue
            elapsed = time.time() - start_time
            logger.info(f"Loaded {len(self.embeddings):,} embeddings in {elapsed:.1f} seconds")
            return True
        except Exception as e:
            logger.error(f"Failed to load embeddings: {e}")
            return False


class EfficientSemanticGraph:
    def __init__(self, vocabulary, embeddings, k=20):
        self.vocabulary = list(vocabulary)
        self.embeddings = embeddings
        self.k = k
        self.graph = nx.Graph()
        self.centrality = {}

    def build(self):
        words_with_embeddings = [w for w in self.vocabulary if w in self.embeddings.embeddings]
        if len(words_with_embeddings) < 50:
            logger.error("Too few words with embeddings")
            return False
        logger.info(f"Building KNN graph with {len(words_with_embeddings)} words, k={self.k}")
        vectors = np.stack([self.embeddings.embeddings[w] for w in words_with_embeddings])
        start_time = time.time()
        nbrs = NearestNeighbors(n_neighbors=self.k, metric="cosine")
        nbrs.fit(vectors)
        distances, indices = nbrs.kneighbors(vectors)
        self.graph.add_nodes_from(words_with_embeddings)
        edges_added = 0
        for i, word in enumerate(words_with_embeddings):
            for j_idx, dist in zip(indices[i][1:], distances[i][1:]):
                neighbor = words_with_embeddings[j_idx]
                similarity = 1 - dist
                if similarity > 0.45:
                    self.graph.add_edge(word, neighbor, weight=similarity, distance=dist)
                    edges_added += 1
        elapsed = time.time() - start_time
        logger.info(f"KNN graph built in {elapsed:.1f}s: {self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges")
        logger.info("Calculating centrality metrics...")
        self.centrality = nx.degree_centrality(self.graph)
        return self.graph.number_of_edges() > 0

    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump({'graph': self.graph, 'vocabulary': self.vocabulary, 'centrality': self.centrality}, f)
        logger.info(f"Graph saved to {filepath}")

    def load(self, filepath):
        try:
            with open(filepath, 'rb') as f:
                data = pickle.load(f)
                self.graph = data['graph']
                self.vocabulary = data['vocabulary']
                self.centrality = data.get('centrality', {})
            logger.info(f"Graph loaded: {self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges")
            return True
        except Exception as e:
            logger.info(f"Could not load graph: {e}")
            return False


class ImprovedClueGenerator:
    def __init__(self, graph):
        self.graph = graph
        self.max_path_length = 3
        self.path_cache = {}

    def find_shortest_path(self, start, end):
        cache_key = (start.lower(), end.lower())
        if cache_key in self.path_cache:
            return self.path_cache[cache_key]
        start, end = start.lower(), end.lower()
        if start not in self.graph.graph or end not in self.graph.graph:
            self.path_cache[cache_key] = None
            return None
        try:
            path = nx.shortest_path(self.graph.graph, start, end, weight='distance')
            path_length = nx.shortest_path_length(self.graph.graph, start, end, weight='distance')
            edge_weights = []
            for i in range(len(path) - 1):
                edge_data = self.graph.graph.get_edge_data(path[i], path[i+1])
                if edge_data:
                    edge_weights.append(edge_data['weight'])
            min_weight = min(edge_weights) if edge_weights else 0.0
            result = SemanticPath(start_word=start, end_word=end, path=path,
                                  total_distance=path_length, path_strength=min_weight, hops=len(path)-1)
            self.path_cache[cache_key] = result
            return result
        except nx.NetworkXNoPath:
            self.path_cache[cache_key] = None
            return None

    def calculate_safety_score(self, clue, target_words, opponent_words, neutral_words, assassin_words):
        if clue not in self.graph.graph:
            return -2.0
        try:
            distances = nx.single_source_dijkstra_path_length(self.graph.graph, clue, weight="distance")
        except:
            return -2.0
        target_distances = [distances.get(w, 999) for w in target_words]
        if not target_distances or min(target_distances) > 3.0:
            return -3.0
        safety_score = 0.0
        enemy_distances = [distances.get(w, 999) for w in opponent_words + neutral_words]
        if enemy_distances:
            min_enemy_dist = min(enemy_distances)
            max_target_dist = max(target_distances)
            safety_margin = min_enemy_dist - max_target_dist
            if safety_margin < -0.5:
                safety_score -= 5.0
            elif safety_margin > 0:
                safety_score += safety_margin * 2.0
        assassin_distances = [distances.get(w, 999) for w in assassin_words]
        if assassin_distances:
            min_assassin_dist = min(assassin_distances)
            if min_assassin_dist < 1.0:
                safety_score -= 5.0
            else:
                safety_score += 1.0
        centrality_bonus = 2.0 * self.graph.centrality.get(clue, 0)
        safety_score += centrality_bonus
        return safety_score

    def generate_clues(self, target_words, opponent_words, neutral_words, assassin_words):
        target_words = [w.lower() for w in target_words]
        opponent_words = [w.lower() for w in opponent_words]
        neutral_words = [w.lower() for w in neutral_words]
        assassin_words = [w.lower() for w in assassin_words]
        all_board_words = set(target_words + opponent_words + neutral_words + assassin_words)

        logger.info(f"Generating clues for: {target_words}")

        candidates = set()
        for target in target_words:
            if target in self.graph.graph:
                direct_neighbors = list(self.graph.graph.neighbors(target))
                strong_neighbors = []
                for neighbor in direct_neighbors:
                    if self.graph.graph.has_edge(target, neighbor):
                        edge_data = self.graph.graph.get_edge_data(target, neighbor)
                        if edge_data.get('weight', 0) > 0.45:
                            strong_neighbors.append(neighbor)
                candidates.update(strong_neighbors[:20])
                for strong_neighbor in strong_neighbors[:8]:
                    if strong_neighbor in self.graph.graph:
                        second_hop = list(self.graph.graph.neighbors(strong_neighbor))[:12]
                        candidates.update(second_hop)

        valid_candidates = [
            word for word in candidates
            if (word not in all_board_words and word not in target_words and
                len(word) >= 3 and len(word) <= 12 and word.isalpha())
        ]

        logger.info(f"Testing {len(valid_candidates)} clue candidates")

        clue_candidates = []
        for clue_word in valid_candidates[:150]:
            paths = {}
            strengths = []
            for target in target_words:
                path = self.find_shortest_path(clue_word, target)
                if path and path.hops <= 3:
                    paths[target] = path
                    strengths.append(path.path_strength)

            min_targets = 1 if len(clue_candidates) < 5 else 2
            if len(paths) < min_targets:
                continue

            avg_strength = np.mean(strengths)
            coverage = len(paths)
            consistency = 1.0 - np.std(strengths) if len(strengths) > 1 else 1.0

            direct_bonus = 0.0
            for path in paths.values():
                if path.hops == 1:
                    direct_bonus += 1.0
                elif path.path_strength > 0.7:
                    direct_bonus += 0.5

            weak_penalty = sum(-0.5 for p in paths.values() if p.path_strength < 0.6)

            creativity_score = (coverage * 0.5 + avg_strength * 0.3 +
                               consistency * 0.2 + direct_bonus + weak_penalty)

            safety_score = self.calculate_safety_score(
                clue_word, list(paths.keys()), opponent_words, neutral_words, assassin_words
            )

            final_score = creativity_score * 0.7 + safety_score * 0.3

            if safety_score > -3.0:
                candidate = ClueCandidate(
                    clue_word=clue_word, target_words=list(paths.keys()), paths=paths,
                    avg_path_strength=avg_strength, coverage=coverage,
                    safety_score=safety_score, creativity_score=creativity_score,
                    final_score=final_score
                )
                clue_candidates.append(candidate)

        clue_candidates.sort(key=lambda x: x.final_score, reverse=True)
        logger.info(f"Generated {len(clue_candidates)} clue candidates")
        return clue_candidates[:10]


class ImprovedCodenames:

    def __init__(self, wordlist_file=None):
        self.vocabulary = Vocabulary(wordlist_file)
        self.embeddings = Embeddings()
        self.graph = None
        self.generator = None
        self.setup_complete = False

    def setup(self, glove_file):
        logger.info("=== Codenames Engine Setup ===")
        if not self.embeddings.load_glove(glove_file):
            return False
        expanded_vocab = self.vocabulary.expand_vocabulary(self.embeddings.embeddings)
        self.graph = EfficientSemanticGraph(expanded_vocab, self.embeddings, k=20)
        cache_file = "codenames_graph_v4.pkl"
        if not self.graph.load(cache_file):
            logger.info("Building KNN semantic graph...")
            if self.graph.build():
                self.graph.save(cache_file)
            else:
                return False
        self.generator = ImprovedClueGenerator(self.graph)
        self.setup_complete = True
        logger.info("=== Engine Setup Complete ===")
        return True

    def generate_clue(self, own_words, opponent_words, neutral_words, assassin_words, revealed_words=None):
        if not self.setup_complete:
            raise RuntimeError("Must call setup() first")
        if revealed_words is None:
            revealed_words = []
        targets = [w for w in own_words if w not in revealed_words]
        if not targets:
            return "pass", 0, []
        candidates = self.generator.generate_clues(targets, opponent_words, neutral_words, assassin_words)
        if not candidates:
            return "pass", 0, []
        best = candidates[0]
        return best.clue_word, best.coverage, best.target_words

In [ ]:
import os
import sys
import re
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

@dataclass
class BoardDef:
    board_id: str
    level: int
    our: List[str]
    opp: List[str]
    assassin: List[str]
    neutral: List[str]

    @property
    def all_words(self) -> List[str]:
        return self.our + self.opp + self.assassin + self.neutral


def parse_boards(filepath: str) -> List[BoardDef]:
    boards = []
    current = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue

            if line.startswith('BOARD '):
                board_id = line.split()[1]
                level = int(board_id.split('.')[0])
                current = BoardDef(
                    board_id=board_id, level=level,
                    our=[], opp=[], assassin=[], neutral=[]
                )
                boards.append(current)

            elif current is not None:
                if line.startswith('OUR:'):
                    current.our = [w.strip().lower() for w in line[4:].split(',')]
                elif line.startswith('OPP:'):
                    current.opp = [w.strip().lower() for w in line[4:].split(',')]
                elif line.startswith('ASSASSIN:'):
                    current.assassin = [w.strip().lower() for w in line[9:].split(',')]
                elif line.startswith('NEUTRAL:'):
                    current.neutral = [w.strip().lower() for w in line[8:].split(',')]

    return boards

class SimpleGuesser:
    def __init__(self, embeddings):
        self.embeddings = embeddings

    def _cosine_sim(self, a, b):
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na == 0 or nb == 0:
            return 0.0
        return float(np.dot(a, b) / (na * nb))

    def guess(self, clue: str, num: int, board_words: List[str]) -> List[str]:
        clue = clue.lower()
        if clue not in self.embeddings:
            import random
            return random.sample(board_words, min(num, len(board_words)))
        clue_vec = self.embeddings[clue]
        scored = []
        for word in board_words:
            w = word.lower()
            if w in self.embeddings:
                scored.append((word, self._cosine_sim(clue_vec, self.embeddings[w])))
            else:
                scored.append((word, -1.0))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [w for w, _ in scored[:num]]

@dataclass
class GameResult:
    board_id: str
    level: int
    turns: int = 0
    correct_guesses: int = 0
    total_guesses: int = 0
    assassin_hit: bool = False
    clue_sizes: List[int] = field(default_factory=list)
    clue_log: List[dict] = field(default_factory=list)


def play_board(board: BoardDef, engine: ImprovedCodenames, guesser: SimpleGuesser,
               verbose: bool = False) -> GameResult:
    result = GameResult(board_id=board.board_id, level=board.level)

    remaining_our = set(board.our)
    remaining_opp = set(board.opp)
    remaining_neutral = set(board.neutral)
    remaining_assassin = set(board.assassin)
    revealed = set()
    used_clues = set()

    max_turns = 15

    if verbose:
        print(f"\n{'─'*50}")
        print(f"Board {board.board_id} | OUR: {board.our}")
        print(f"  OPP: {board.opp} | ASSASSIN: {board.assassin}")

    while remaining_our and result.turns < max_turns:
        result.turns += 1

        our_list = list(remaining_our)
        opp_list = list(remaining_opp)
        neutral_list = list(remaining_neutral)
        assassin_list = list(remaining_assassin)

        clue, num, intended = engine.generate_clue(
            own_words=our_list,
            opponent_words=opp_list,
            neutral_words=neutral_list,
            assassin_words=assassin_list,
            revealed_words=list(revealed)
        )

        if clue in used_clues or clue == "pass" or num == 0:
            if verbose:
                print(f"  Turn {result.turns}: PASS (no new clue)")
            if result.turns >= 3 and all(
                len(result.clue_log) >= i and
                result.clue_log[-(i)]['clue'] == 'pass'
                for i in range(1, min(4, len(result.clue_log) + 1))
                if len(result.clue_log) >= i
            ):
                break
            continue

        used_clues.add(clue)
        result.clue_sizes.append(num)

        unrevealed = sorted(
            list(remaining_our | remaining_opp | remaining_neutral | remaining_assassin)
        )
        guesses = guesser.guess(clue, num, unrevealed)

        turn_info = {
            'turn': result.turns, 'clue': clue, 'num': num,
            'intended': intended, 'guesses': guesses, 'outcomes': []
        }

        if verbose:
            print(f"  Turn {result.turns}: Clue=\"{clue.upper()}\" for {num} → guesses: {guesses}")

        for g in guesses:
            g_lower = g.lower()
            result.total_guesses += 1

            if g_lower in remaining_assassin:
                result.assassin_hit = True
                turn_info['outcomes'].append((g, 'ASSASSIN'))
                if verbose:
                    print(f"    💀 '{g}' = ASSASSIN! Game over.")
                result.clue_log.append(turn_info)
                return result

            elif g_lower in remaining_our:
                result.correct_guesses += 1
                remaining_our.discard(g_lower)
                revealed.add(g_lower)
                turn_info['outcomes'].append((g, 'CORRECT'))
                if verbose:
                    print(f"    ✅ '{g}' = OUR ({result.correct_guesses}/8)")

            elif g_lower in remaining_opp:
                remaining_opp.discard(g_lower)
                revealed.add(g_lower)
                turn_info['outcomes'].append((g, 'OPPONENT'))
                if verbose:
                    print(f"    ❌ '{g}' = OPPONENT. Turn ends.")
                break

            elif g_lower in remaining_neutral:
                remaining_neutral.discard(g_lower)
                revealed.add(g_lower)
                turn_info['outcomes'].append((g, 'NEUTRAL'))
                if verbose:
                    print(f"    ⬜ '{g}' = NEUTRAL. Turn ends.")
                break

        result.clue_log.append(turn_info)

    if verbose:
        status = "ASSASSIN" if result.assassin_hit else ("WON" if not remaining_our else "MAX TURNS")
        print(f"  Result: {status} | {result.correct_guesses}/8 found in {result.turns} turns")

    return result


@dataclass
class LevelMetrics:
    level: int
    avg_turns: float = 0.0
    accuracy: float = 0.0
    avg_clue_size: float = 0.0
    assassin_rate: float = 0.0


def aggregate_metrics(results: List[GameResult]) -> Dict[int, LevelMetrics]:
    from collections import defaultdict

    by_level = defaultdict(list)
    for r in results:
        by_level[r.level].append(r)

    metrics = {}
    for level in sorted(by_level.keys()):
        games = by_level[level]
        n = len(games)

        avg_turns = sum(g.turns for g in games) / n
        total_correct = sum(g.correct_guesses for g in games)
        total_guesses = sum(g.total_guesses for g in games)
        accuracy = (total_correct / total_guesses * 100) if total_guesses > 0 else 0.0

        all_clue_sizes = []
        for g in games:
            all_clue_sizes.extend(g.clue_sizes)
        avg_clue_size = (sum(all_clue_sizes) / len(all_clue_sizes)) if all_clue_sizes else 0.0

        assassin_count = sum(1 for g in games if g.assassin_hit)
        assassin_rate = assassin_count / n * 100

        metrics[level] = LevelMetrics(
            level=level,
            avg_turns=avg_turns,
            accuracy=accuracy,
            avg_clue_size=avg_clue_size,
            assassin_rate=assassin_rate
        )

    return metrics


def print_results_table(metrics: Dict[int, LevelMetrics]):
    print()
    print("=" * 72)
    print("                   CODENAMES EVALUATION RESULTS")
    print("=" * 72)
    print(f"{'Level':>7} │ {'Description':<28} │ {'Turns':>5} │ {'Acc%':>6} │ {'Clue':>4} │ {'☠ %':>5}")
    print("─" * 72)

    descriptions = {
        1: "Tight Cluster (Easy)",
        2: "Two Easy Clusters",
        3: "Three Clusters (3+3+2)",
        4: "Mixed Easy (cluster+outlier)",
        5: "Diverse (8 diff topics)",
        6: "Opponent Distraction",
        7: "Heavy Overlap",
        8: "Assassin Adjacent",
        9: "Assassin Trap",
        10: "Nightmare",
    }

    total_turns = []
    total_acc_num = 0
    total_acc_den = 0
    total_clue_sizes = []
    total_assassin = 0
    total_games = 0

    for level in sorted(metrics.keys()):
        m = metrics[level]
        desc = descriptions.get(level, "")
        print(f"  {level:>5} │ {desc:<28} │ {m.avg_turns:>5.1f} │ {m.accuracy:>5.1f}% │ {m.avg_clue_size:>4.1f} │ {m.assassin_rate:>4.0f}%")
        total_turns.append(m.avg_turns)

    print("─" * 72)

    all_levels = list(metrics.values())
    n = len(all_levels)
    if n > 0:
        overall_turns = sum(m.avg_turns for m in all_levels) / n
        overall_acc = sum(m.accuracy for m in all_levels) / n
        overall_clue = sum(m.avg_clue_size for m in all_levels) / n
        overall_assassin = sum(m.assassin_rate for m in all_levels) / n
        print(f"{'OVERALL':>8} │ {'(averaged across levels)':<28} │ {overall_turns:>5.1f} │ {overall_acc:>5.1f}% │ {overall_clue:>4.1f} │ {overall_assassin:>4.0f}%")

    print("=" * 72)
    print()
    print("Metrics:")
    print("  Turns    = Avg turns to find all 8 OUR words (or game end)")
    print("  Acc%     = Correct guesses / total guesses × 100")
    print("  Clue     = Avg words per clue (clue size)")
    print("  ☠ %      = % of games ending in assassin hit")
    print()



def main():
    verbose = "--verbose" in sys.argv or "-v" in sys.argv

    glove_file = None
    for f in ["glove.6B.100d.txt", "glove.6B.50d.txt"]:
        if os.path.exists(f):
            glove_file = f
            break
    if not glove_file:
        print("GloVe file not found. Download glove.6B.zip first.")
        return

    boards_file = None
    for f in ["difficulty_boards.txt", "/mnt/user-data/uploads/difficulty_boards.txt"]:
        if os.path.exists(f):
            boards_file = f
            break
    if not boards_file:
        print("difficulty_boards.txt not found.")
        return

    boards = parse_boards(boards_file)
    print(f"Loaded {len(boards)} boards across {len(set(b.level for b in boards))} levels")

    engine = ImprovedCodenames()
    if not engine.setup(glove_file):
        print("Engine setup failed.")
        return

    guesser = SimpleGuesser(engine.embeddings.embeddings)

    results = []
    for i, board in enumerate(boards):
        print(f"\nPlaying board {board.board_id} ({i+1}/{len(boards)})...")
        result = play_board(board, engine, guesser, verbose=verbose)
        results.append(result)

        status = "☠ ASSASSIN" if result.assassin_hit else f"{result.correct_guesses}/8"
        print(f"  → {status} in {result.turns} turns "
              f"(accuracy: {result.correct_guesses}/{result.total_guesses})")

    metrics = aggregate_metrics(results)
    print_results_table(metrics)

In [ ]:
main()